# Week 2 Day 3

## Covered Today
1. Building Chat UIs with Gradio: Your First Conversational AI Assistant
2. Building a Streaming Chatbot with Gradio and OpenAI API
3. System Prompts, Multi-Shot Prompting, and Your First Look at RAG

In [3]:
import os
from dotenv import load_dotenv
import requests
from openai import OpenAI
from IPython.display import Markdown, display, update_display
import gradio as gr 

### Now we load all the API Keys

In [4]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

### Now check if all loaded keys exist and are as per the format

In [5]:
if openai_api_key:
    if openai_api_key.startswith("sk-"):
        print(f"OpenAI      : OK           (begins {openai_api_key[:7]}...)")
    else:
        print("OpenAI      : WRONG FORMAT (should start with 'sk-')")
else:
    print("OpenAI      : MISSING")


if anthropic_api_key:
    if anthropic_api_key.startswith("sk-ant-"):
        print(f"Anthropic   : OK           (begins {anthropic_api_key[:10]}...)")
    else:
        print("Anthropic   : WRONG FORMAT (should start with 'sk-ant-')")
else:
    print("Anthropic   : MISSING")


if google_api_key:
    if google_api_key.startswith("AQ.Ab"):
        print(f"Google      : OK           (begins {google_api_key[:5]}...)")
    else:
        print("Google      : WRONG FORMAT (should start with 'AQ.Ab' or 'AIza')")
else:
    print("Google      : MISSING")


if openrouter_api_key:
    if openrouter_api_key.startswith("sk-or-"):
        print(f"OpenRouter  : OK           (begins {openrouter_api_key[:8]}...)")
    else:
        print("OpenRouter  : WRONG FORMAT (should start with 'sk-or-')")
else:
    print("OpenRouter  : MISSING")

OpenAI      : OK           (begins sk-proj...)
Anthropic   : OK           (begins sk-ant-api...)
Google      : OK           (begins AQ.Ab...)
OpenRouter  : OK           (begins sk-or-v1...)


In [6]:
# create clients for each provider
# for openai, we simply use OpenAI, for others we need to specify the base url and key, while using openai api library
openai_client = OpenAI()

google_url = 'https://generativelanguage.googleapis.com/v1beta/openai/'
anthropic_url = 'https://api.anthropic.com/v1/'
openrouter_url = 'https://openrouter.ai/api/v1'
ollama_url = 'http://127.0.0.1:11434/v1'

In [7]:
google_client = OpenAI(base_url=google_url, api_key=google_api_key)
anthropic_client = OpenAI(base_url=anthropic_url, api_key=anthropic_api_key)
openrouter_client = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama_client = OpenAI(base_url=ollama_url, api_key='Ollama')

We start by creating a chat interface on gradio first, with a simple call back function

In [8]:
def chat(message, history):
    return 'Bananas'

In [13]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [14]:
# now we also see how history works, we rewrite the chat callback function again to see it in chat
def chat(message, history):
    return f"Your message is {message}, history is {history}, but I will still say bananas"

In [15]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


so, the first response that we got was: 
```
Your message is hi, history is [], but I will still say bananas
```
then, the next response was
```
Your message is hello, history is [{'role': 'user', 'metadata': None, 'content': [{'text': 'hi', 'type': 'text'}], 'options': None}, {'role': 'assistant', 'metadata': None, 'content': [{'text': 'Your message is hi, history is [], but I will still say bananas', 'type': 'text'}], 'options': None}], but I will still say bananas
```

we tried it once again, and this is what we got: 
```
Your message is hello again, history is [{'role': 'user', 'metadata': None, 'content': [{'text': 'hi', 'type': 'text'}], 'options': None}, {'role': 'assistant', 'metadata': None, 'content': [{'text': 'Your message is hi, history is [], but I will still say bananas', 'type': 'text'}], 'options': None}, {'role': 'user', 'metadata': None, 'content': [{'text': 'hello', 'type': 'text'}], 'options': None}, {'role': 'assistant', 'metadata': None, 'content': [{'text': "Your message is hello, history is [{'role': 'user', 'metadata': None, 'content': [{'text': 'hi', 'type': 'text'}], 'options': None}, {'role': 'assistant', 'metadata': None, 'content': [{'text': 'Your message is hi, history is [], but I will still say bananas', 'type': 'text'}], 'options': None}], but I will still say bananas", 'type': 'text'}], 'options': None}], but I will still say bananas
```

as we can see, gradio is passing in the history of the chat, very similar to the openai messages format and we can use this to our advantage to send the conversation history to our chat LLM API, so as to give an illusion of memory in the chatinterface as we see in LLM tools such as chatgpt or gemini.

In [43]:
# using this, we can create a function, which extracts this history and then pass it across in messages, so that the LLM is aware of the dicsussion
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": "You are a helpful assistant"}] \
    + history \
    + [{"role": "user", "content": message}]

    response = openai_client.chat.completions.create(model='gpt-4.1-mini', messages=messages) #type: ignore
    print(f"History:\n{history}\n\n")
    print(f"Message:\n{message}\n\n")
    return response.choices[0].message.content

In [44]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7883
* To create a public link, set `share=True` in `launch()`.


History:
[]


Message:
hi


History:
[{'role': 'user', 'content': [{'text': 'hi', 'type': 'text'}]}, {'role': 'assistant', 'content': [{'text': 'Hello! How can I assist you today?', 'type': 'text'}]}]


Message:
tell me about moon


History:
[{'role': 'user', 'content': [{'text': 'hi', 'type': 'text'}]}, {'role': 'assistant', 'content': [{'text': 'Hello! How can I assist you today?', 'type': 'text'}]}, {'role': 'user', 'content': [{'text': 'tell me about moon', 'type': 'text'}]}, {'role': 'assistant', 'content': [{'text': "Sure! Here are some interesting facts about the Moon:\n\n1. **Natural Satellite:** The Moon is Earth's only natural satellite and the fifth largest moon in the solar system.\n\n2. **Distance from Earth:** It is about 384,400 kilometers (238,855 miles) away from Earth.\n\n3. **Size:** The Moon's diameter is about 3,474 kilometers (2,159 miles), which is roughly 27% the size of Earth.\n\n4. **Gravity:** The gravity on the Moon is about 1/6th of Earth's gravity.\n\n5. *